In [1]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

response = client.chat.completions.create(
    model="qwen2.5:3b",
    messages=[
        {
            "role": "user",
            "content": "你好，请用一句话介绍 Transformer。"
        }
    ],
)

print(response.choices[0].message.content)

Transformer是一种用于序列到序列学习的神经网络架构，特别适用于自然语言处理任务。


#### 1：Function Calling（一个工具、一次调用）

In [7]:
def get_weather(city):
    fake_weather = {
        "北京": "晴",
        "台北": "小雨",
        "上海": "阴",
    }

    return fake_weather.get(city, "未知")

In [10]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

weather_tool = {"type": "function", 
                "function": {"name": "get_weather", 
                             "description": "查询城市目前天气（晴/雨/阴），回传一个短字符串。",
                             "parameters": {"type": "object", 
                                            "properties": {"city": {"type": "string", "description": "城市名称。"}},
                            "required": ["city"],
                            }
                             }
                }
res = client.chat.completions.create(model="qwen2.5:3b", max_tokens=512, tools=[weather_tool], messages=[{"role": "user", "content":"北京现在下雨吗？"}])

msg = res.choices[0].message
print("conten: ", msg.content)
print("finish_reason: ", res.choices[0].finish_reason)
print("tool_calls: ", msg.tool_calls)
import json
assert msg.tool_calls, "预期 LLM 会选择调用 tool（而非直接回答）"
tc = msg.tool_calls[0]
assert tc.function.name == "get_weather", f"预期调用 get_weather、实际 {tc.function.name}"
args = json.loads(tc.function.arguments)
print(f"args: {args}")
assert args.get("city"), "预期 city 参数有值"
print(f"✅ 练习 1 通过 — qwen2.5:3b 正确选了 get_weather、带 city='{args['city']}' 参数")

conten:  
finish_reason:  tool_calls
tool_calls:  [ChatCompletionMessageFunctionToolCall(id='call_ewnk3pt1', function=Function(arguments='{"city":"北京"}', name='get_weather'), type='function', index=0)]
args: {'city': '北京'}
✅ 练习 1 通过 — qwen2.5:3b 正确选了 get_weather、带 city='北京' 参数


In [9]:
args

{'city': '北京'}